# ATLAS — LM ~100M · Texto (3 mixers)

**Self-contained** — testa **texto** no Colab sem esperar batch LM na VM.

**Como usar:**
1. Runtime → **GPU (A100)**
2. **Run all** (~**45–90 min** total)

**3 configs × 1 seed** (rápido; aumente seeds depois):

| ID | Mixer | O que testa |
|----|-------|-------------|
| `lm-100m-base` | global-attn | Baseline transformer |
| `lm-100m-tb` | token_blend | TokenBlend causal O(T) |
| `lm-100m-tbms` | token_blend_ms | Multi-scale causal |

**Escala:** ~100M params · Shakespeare char-LM · 4000 steps · bf16

**Métrica:** `val_loss` ↓ = melhor


In [ ]:
# @title 1. GPU
import torch
assert torch.cuda.is_available(), 'Ative GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


In [ ]:
# @title 2. Bundle ATLAS LM (sem GitHub)
import base64, gzip, io, sys, tarfile, importlib
from pathlib import Path

BUNDLE_B64 = """..."""

raw = gzip.decompress(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(".")

sys.path.insert(0, str(Path(".").resolve()))
for mod in list(sys.modules):
    if mod == "atlas" or mod.startswith("atlas."):
        del sys.modules[mod]
import atlas.models, atlas.datasets, atlas.trainer, atlas.optimizers
from atlas.trainer import LMTrainConfig, train_lm
from atlas.models import CharLM
from dataclasses import asdict

print("Bundle OK — shakespeare", Path("atlas/data/tinyshakespeare.txt").stat().st_size, "bytes")


In [ ]:
# @title 3. Verificar ~100M params
from atlas.datasets import load_train_val
train_ds, _ = load_train_val('atlas/data/tinyshakespeare.txt')
cfg = LMTrainConfig(n_layer=12, n_head=8, n_embd=832, block_size=256)
m = CharLM(vocab_size=train_ds.vocab_size, n_layer=cfg.n_layer, n_head=cfg.n_head,
           n_embd=cfg.n_embd, block_size=cfg.block_size, dropout=cfg.dropout, mixing='global-attn')
n = sum(p.numel() for p in m.parameters())
print(f'Params: {n/1e6:.2f}M (alvo ~100M)')
assert 90 <= n/1e6 <= 115


In [ ]:
# @title 4. Treinar 3 mixers @~100M
import json, time
from pathlib import Path
from datetime import datetime, timezone

BASE = dict(
    n_layer=12, n_head=8, n_embd=832, block_size=256,
    batch=32, dropout=0.1, lr=3e-4, weight_decay=0.1,
    total_steps=4000, eval_every=500, log_every=500,
    warmup_steps=200, schedule='cosine-warmup',
    optimizer='adamw', device='cuda', amp=True,
    data_path='atlas/data/tinyshakespeare.txt',
)
EXPERIMENTS = [
    ('lm-100m-base', 'baseline_attn', 'global-attn'),
    ('lm-100m-tb', 'token_blend', 'token_blend'),
    ('lm-100m-tbms', 'token_blend_ms', 'token_blend_ms'),
]
RESULT_DIR = Path('results/lm_colab')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
LOG_ENTRIES = []
T0 = time.time()

for eid, name, mixing in EXPERIMENTS:
    print(f'\n{"="*60}\n{eid} | {mixing}\n{"="*60}')
    t0 = time.time()
    cfg = LMTrainConfig(tag=name, seed=42, mixing=mixing, **BASE)
    result = train_lm(cfg)
    payload = asdict(result)
    out = RESULT_DIR / f'{eid}_seed42.json'
    out.write_text(json.dumps(payload, indent=2)+'\n')
    entry = {
        'id': eid, 'name': name, 'mixing': mixing,
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'result': payload,
    }
    LOG_ENTRIES.append(entry)
    print(f"  val_loss={result.best_val_loss:.4f}  ({result.steps_per_sec:.1f} steps/s)  ({time.time()-t0:.0f}s)")

log_path = Path('experiments_log_lm_colab.jsonl')
with log_path.open('w') as f:
    for e in LOG_ENTRIES:
        f.write(json.dumps(e, ensure_ascii=False)+'\n')
print(f'\nSalvo: {log_path}  | total { (time.time()-T0)/60:.1f} min')


In [ ]:
# @title 5. Ranking + amostra de texto
import torch.nn.functional as F

@torch.no_grad()
def generate(model, stoi, itos, prompt, max_new=200, temperature=0.8, device='cuda'):
    model.eval().to(device)
    idx = torch.tensor([[stoi[c] for c in prompt]], device=device)
    for _ in range(max_new):
        ctx = idx[:, -model.block_size:]
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)
    return ''.join(itos[i] for i in idx[0].tolist())

rows = sorted(LOG_ENTRIES, key=lambda e: e['result']['best_val_loss'])
print(f"{'Rank':<5} {'ID':<14} {'val_loss':>10} {'mixer':<16}")
print('-'*50)
for i, e in enumerate(rows, 1):
    print(f"{i:<5} {e['id']:<14} {e['result']['best_val_loss']:>10.4f} {e['mixing']:<16}")

best = rows[0]
print(f"\n🏆 Melhor mixer: {best['id']} (loss {best['result']['best_val_loss']:.4f})")

# amostra rápida do melhor (re-treino curto só p/ demo — usa último modelo em memória)
print('\n--- Amostra do último treino (prompt Shakespeare) ---')
prompt = 'First Citizen:\n'
try:
    sample = generate(m, train_ds.stoi, train_ds.itos, prompt, max_new=150)
    print(sample[:500])
except Exception as ex:
    print('(amostra opcional falhou — ranking acima é o que importa)', ex)


In [ ]:
# @title 6. Download
from google.colab import files
files.download('experiments_log_lm_colab.jsonl')
